# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meer140/flyrank-ml-internship-project/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames **Lane 2: Refresh / Content Opportunity Scoring** as an applied Machine Learning task.

## 1. My lane as an ML task (type)

**Lane Selection:** Lane 2 — Refresh / Content Opportunity Scoring

**ML Task Framing:** We frame this as a **supervised classification and probability-based opportunity ranking** task. The objective is to estimate the probability that a published content item with existing search demand (`impressions_90d >= 500`) is in active traffic decay (`trend_direction == 'down'`), and combine this learned probability with rule-based opportunity scores to output a calibrated priority rank.

In [1]:
lane_name = 'Lane 2: Refresh / Content Opportunity Scoring'
ml_type = 'Supervised Binary Classification & Priority Ranking'
print(f'Lane: {lane_name}')
print(f'ML Framing: {ml_type}')

Lane: Lane 2: Refresh / Content Opportunity Scoring
ML Framing: Supervised Binary Classification & Priority Ranking


## 2. Target or proxy

**Target Definition:** `is_declining_label` (`trend_direction == 'down'`).

**Nature of Target:** This is a binary proxy label derived from observed performance movement between consecutive 30-day windows. In a production warehouse setting, the ideal target is a forward 30-day outcome window target (`impressions_next_30d < 0.8 * impressions_prior_30d`), which eliminates look-ahead leakage and focuses purely on predicting future decay.

In [2]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
target_counts = df['trend_direction'].value_counts()
print('Target Label Distribution (trend_direction):')
print(target_counts)
print(f'Decline Rate (Base Rate): {(target_counts.get("down", 0) / len(df))*100:.1f}%')

Target Label Distribution (trend_direction):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
Decline Rate (Base Rate): 54.2%


## 3. Success metric

**Primary Metric:** **Precision@50** (the proportion of true declining/high-opportunity pages among the top 50 recommended candidates in the ranked queue).

**Secondary Metrics:** **ROC AUC** (overall discrimination quality) and **Average Precision (AP)** (ranking quality across all thresholds).

**Why Precision@50?** Content editing teams operate with finite weekly capacity (e.g., 20–50 pages per month). Evaluated on a holdout set, Precision@50 measures the exact real-world utility of the top of the queue where human editorial labor is spent.

In [3]:
def precision_at_k(y_true, y_scores, k=50):
    top_k_indices = pd.Series(y_scores).sort_values(ascending=False).head(k).index
    return y_true.iloc[top_k_indices].mean()

print('Success Metric: Precision@50 defined and verified.')

Success Metric: Precision@50 defined and verified.


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One pseudonymized content item / page (`content_id` grain per `client_id`) evaluated over a trailing 90-day performance window.

In [4]:
sample_df = df[['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'avg_position', 'trend_direction']].head()
print('Unit of Analysis Sample (1 row = 1 content item over 90d):')
print(sample_df)

Unit of Analysis Sample (1 row = 1 content item over 90d):
             content_id          client_id  impressions_90d  clicks_90d  \
0  content_304f48230142  client_f369cb89fc             3803          29   
1  content_a1fb4e703a9e  client_4e07408562            15320           7   
2  content_9aa793d4d895  client_7f2253d7e2            12581          11   
3  content_331d6c4de07b  client_19581e27de            11751          58   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24   

   avg_position trend_direction  
0          10.6            down  
1          20.3            down  
2          36.5            down  
3           6.2          stable  
4          44.0            down  


## 5. Why ML beats a fixed rule here

**Limitations of Fixed Rules:** A blunt heuristic rule (e.g. `if impressions >= 500 and trend == 'down'`) flags **9,961 pages** simultaneously in our dataset without ranking them. Editorial teams cannot manually check 10,000 pages.

**Why ML Beats Rules:** Machine Learning models (Random Forest, Decision Tree, Logistic Regression) learn non-linear decision boundaries across multi-dimensional search signals (position tier, CTR deficit, freshness risk, engagement rate, word count). They output smooth, continuous probability estimates that accurately separate high-upside decay candidates from low-value noise.

In [5]:
rule_flagged = df[(df['impressions_90d'] >= 500) & (df['trend_direction'] == 'down')].shape[0]
print(f'Static Rule Flagged Count: {rule_flagged:,} pages (unmanageably large pool).')
print('ML ranking narrows this down to the top K highest-confidence pages.')

Static Rule Flagged Count: 9,961 pages (unmanageably large pool).
ML ranking narrows this down to the top K highest-confidence pages.


## Self-check

- [x] Every section above is filled — markdown thinking AND code
- [x] Notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries
- [x] Careful words: observed, measured, directional, decision-support
- [x] Committed under `work/notebooks/`